# RAG sobre los manuales CRP — comparable con `train_gemma.ipynb` / `evaluate_gemma.ipynb`

Mismo dataset (`qa.jsonl`) y **el mismo split de test** (`seed=42`,
`test_size=0.10`) que usa `evaluate_gemma.ipynb`, para que las respuestas de
este notebook sean directamente comparables contra el modelo **base** y el
modelo **afinado con LoRA** en la misma tabla final.

Pipeline:
1. Cargar `corpus_pages.jsonl` (páginas extraídas de los manuales CRP) y
   trocearlo en chunks más manejables.
2. Indexar los chunks con embeddings (`sentence-transformers`) + FAISS
   (CPU, no consume crédito de GPU — el índice se calcula una sola vez).
3. Para cada pregunta del split de test: recuperar los `k` chunks más
   relevantes, armar un prompt con contexto + pregunta, y generar la
   respuesta con `google/gemma-7b-it` **sin fine-tuning** (RAG puro).
4. Calcular ROUGE / BERTScore igual que en `evaluate_gemma.ipynb`.
5. Fusionar con el CSV que ya generó `evaluate_gemma.ipynb`
   (`eval_crobotp_base_vs_finetuned.csv`) para construir la tabla
   comparativa final: **base vs. afinado (LoRA) vs. RAG**.


## 0. Instalar dependencias

In [ ]:
%pip install torch transformers accelerate bitsandbytes peft \
    sentence-transformers faiss-cpu rank_bm25 huggingface_hub \
    evaluate rouge_score bert_score absl-py pandas datasets


## 0b. Login en Hugging Face

google/gemma-7b-it es un repo *gated* -necesitas un token de HF con la
licencia del modelo ya aceptada en tu cuenta (el mismo que usaste en
`train_gemma.ipynb`).


In [1]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
login(token=hf_token)
del hf_token  # no lo dejamos flotando en una variable mas de lo necesario


## 1. Configuración

Mismos valores de `DATA_PATH`, `EVAL_SPLIT_SIZE` y `SEED` que
`evaluate_gemma.ipynb` — así el split de test es *idéntico*, ejemplo por
ejemplo, al que se usó para comparar base vs. afinado.


In [2]:
import os

# --- Modelo generador (RAG usa el modelo BASE, sin LoRA) ---
MODEL_NAME = "google/gemma-7b-it"
MAX_NEW_TOKENS = 256

# --- Datos ---
CORPUS_PATH = "./corpus/corpus_pages.jsonl"   # manual, page, text
DATA_PATH = "./corpus/qa.jsonl"               # mismo archivo que el fine-tuning
EVAL_SPLIT_SIZE = 0.10                        # mismo 10% que evaluate_gemma.ipynb
SEED = 42                                     # misma semilla -> mismo split de test

# --- Chunking del corpus ---
CHUNK_SIZE = 800     # caracteres por chunk (algunas paginas tienen +30k chars)
CHUNK_OVERLAP = 100

# --- Retrieval ---
# BGE es un modelo entrenado para retrieval (no solo similitud generica de
# oraciones como MiniLM) -distingue mejor "un chunk que MENCIONA seguridad"
# de "el chunk que SI tiene el dato especifico que se pregunta".
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
# BGE recomienda anteponer esta instruccion SOLO a las preguntas (no a los
# chunks del corpus) para el caso de uso "pregunta corta -> pasaje largo".
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
TOP_K = 6            # cuantos chunks van al prompt final
RETRIEVE_FETCH_K = 20  # candidatos a traer ANTES de deduplicar por pagina

# --- Salidas ---
OUTPUT_CSV = os.environ.get("RAG_OUTPUT_CSV", "/home/jovyan/labs/eval_crobotp_rag.csv")
COMPARISON_CSV = os.environ.get(
    "COMPARISON_CSV", "/home/jovyan/labs/eval_crobotp_base_vs_finetuned_vs_rag.csv"
)
# CSV que ya produjo evaluate_gemma.ipynb (base vs. afinado), para fusionar al final
FINETUNE_EVAL_CSV = os.environ.get(
    "FINETUNE_EVAL_CSV", "/home/jovyan/labs/eval_crobotp_base_vs_finetuned.csv"
)

print(f"Modelo generador: {MODEL_NAME}")
print(f"Corpus: {CORPUS_PATH}")
print(f"QA: {DATA_PATH} (split test: {EVAL_SPLIT_SIZE}, seed={SEED})")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")


Modelo generador: google/gemma-7b-it
Corpus: ./corpus/corpus_pages.jsonl
QA: ./corpus/qa.jsonl (split test: 0.1, seed=42)
Modelo de embeddings: BAAI/bge-small-en-v1.5


**Nota sobre el retrieval (leer antes de correr la seccion 10 completa):**

Con `all-MiniLM-L6-v2` + `TOP_K=4` sin deduplicar, el retriever devolvia 2
chunks de la MISMA pagina y ademas confundia paginas que *hablan de*
seguridad con la pagina que *tiene* el dato especifico -se corrigio
deduplicando por `(manual, page)` y subiendo a `BAAI/bge-small-en-v1.5`.

Aun asi, algunas preguntas (ej. la del estado del motor) siguen fallando:
el manual tiene VARIAS paginas seguidas con el mismo estilo de "advertencia
de seguridad + consecuencia" (paginas 4-7 del Robot Operation Manual), asi
que para el embedding son casi indistinguibles entre si -todas hablan de
"seguridad" con la misma estructura. Por eso se agrega retrieval **hibrido**
(denso + BM25, fusionados con Reciprocal Rank Fusion):

- El componente **denso** (BGE) captura similitud semantica general.
- El componente **BM25** (lexico, por palabras clave) rescata coincidencias
  literales exactas -como "before operating the robot" + "motor" + "status",
  que SI aparecen tal cual en la pagina correcta y no en las paginas vecinas.

Vuelve a correr la seccion 8 (prueba manual) despues de este cambio.

## 2. Verificar GPU\n\nSolo la necesitamos para *generar* con Gemma. El índice FAISS se construye en CPU.

In [3]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: no se detecto GPU. Revisa el contenedor/VM (ver docker-compose.yml).")


CUDA disponible: False
Aviso: no se detecto GPU. Revisa el contenedor/VM (ver docker-compose.yml).


## 3. Cargar y trocear el corpus (`corpus_pages.jsonl`)

Cada registro del archivo es una página completa de un manual (`manual`,
`page`, `text`). Algunas páginas son muy largas (hay una con ~39.000
caracteres), así que las troceamos en chunks de `CHUNK_SIZE` caracteres con
solapamiento `CHUNK_OVERLAP`, para que el retriever recupere pasajes
puntuales en vez de páginas enteras.


In [3]:
import json

def load_corpus_pages(path):
    pages = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                pages.append(json.loads(line))
    return pages

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    step = max(chunk_size - overlap, 1)
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

pages = load_corpus_pages(CORPUS_PATH)
print(f"Paginas cargadas: {len(pages)}")

chunks = []          # texto de cada chunk
chunk_meta = []       # (manual, page) de cada chunk, para citar la fuente
for p in pages:
    for c in chunk_text(p["text"]):
        chunks.append(c)
        chunk_meta.append({"manual": p["manual"], "page": p["page"]})

print(f"Chunks totales: {len(chunks)}")
print(f"Ejemplo de chunk:\n---\n{chunks[0][:300]}...\n---")


Paginas cargadas: 227
Chunks totales: 695
Ejemplo de chunk:
---
Installation Dimension of Base
INSTALLATION INTERFACE DIAGRAM
MOTION RANGE DIAGRAM
Flange Dimensions
PRODUCT INTRODUCTION 39/40
CRP-RA07A-08 CRP-RA09A-07
CRP-RA07A-08
CRP-RA09A-07
INDUSTRIAL ROBOT
HANDLING 
APPLICATION
· Adopt modular design to effectively reduce the failure rate of the whole machin...
---


## 4. Embeddings + índice FAISS

Se calcula una sola vez (CPU, no requiere GPU ni gasta crédito de Vertex).


In [18]:
!pip install --upgrade huggingface_hub transformers sentence-transformers

  Using cached huggingface_hub-2.0.0-py3-none-any.whl (828 kB)



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_embeddings = embedder.encode(
    chunks, batch_size=64, show_progress_bar=True, normalize_embeddings=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # producto interno sobre vectores normalizados = similitud coseno
index.add(chunk_embeddings)

print(f"Indice FAISS construido: {index.ntotal} vectores de dimension {dim}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\OneDrivePersonal\OneDrive\Maestria\Aprendizaje automatico\venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Indice FAISS construido: 695 vectores de dimension 384


## 4b. Índice BM25 (retrieval híbrido)

BM25 es búsqueda léxica clásica (como la de un buscador de texto completo):
no entiende semántica, pero es muy fuerte encontrando coincidencias
*literales* de palabras clave -exactamente lo que falla en un embedding
cuando varias páginas comparten el mismo tema y estructura.


In [7]:
!pip install rank_bm25


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

tokenized_chunks = [tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"Indice BM25 construido sobre {len(tokenized_chunks)} chunks.")


Indice BM25 construido sobre 695 chunks.


## 5. Función de recuperación (retrieval)

In [6]:
import numpy as np

def rrf_fuse(rank_arrays, k_rrf=60):
    """Reciprocal Rank Fusion: combina varios rankings sin necesitar
    normalizar escalas de score distintas (coseno vs. BM25)."""
    n = len(rank_arrays[0])
    fused = np.zeros(n)
    for scores in rank_arrays:
        order = np.argsort(-scores)          # de mayor a menor score
        ranks = np.empty(n, dtype=int)
        ranks[order] = np.arange(n)          # rank 0 = mejor
        fused += 1.0 / (k_rrf + ranks + 1)
    return fused

def retrieve(query, k=TOP_K):
    # --- score denso (BGE) ---
    q_text = QUERY_INSTRUCTION + query
    q_emb = embedder.encode([q_text], normalize_embeddings=True).astype("float32")[0]
    dense_scores = chunk_embeddings @ q_emb  # coseno, ya normalizado

    # --- score lexico (BM25) ---
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)))

    # --- fusion ---
    fused_scores = rrf_fuse([dense_scores, bm25_scores])
    order = np.argsort(-fused_scores)

    # deduplicar por pagina, igual que antes
    seen_pages = set()
    results = []
    for i in order:
        key = (chunk_meta[i]["manual"], chunk_meta[i]["page"])
        if key in seen_pages:
            continue
        seen_pages.add(key)
        results.append({
            "text": chunks[i],
            "manual": chunk_meta[i]["manual"],
            "page": chunk_meta[i]["page"],
            "score": float(fused_scores[i]),
            "dense_score": float(dense_scores[i]),
            "bm25_score": float(bm25_scores[i]),
        })
        if len(results) == k:
            break
    return results

# Sanity check rapido -misma pregunta que antes fallo, para confirmar el fix
_ejemplo = retrieve("What safety status must the motor be in before operating the robot?")
for r in _ejemplo:
    print(f"[{r['manual']} p.{r['page']}  fused={r['score']:.4f}  "
          f"dense={r['dense_score']:.3f}  bm25={r['bm25_score']:.2f}]  {r['text'][:100]}...")


[EN-00001-A1_Robot_Operation_Manual__1 p.7  fused=0.0323  dense=0.760  bm25=23.02]  ★All the operators of robot systems should participate in system 
training, study safety protection ...
[EN-00001-A1_Robot_Operation_Manual__1 p.27  fused=0.0315  dense=0.760  bm25=21.32]  6. Manual Operation
Before operating industrial robots, please read the CRP Robot 
Safety Manual, CR...
[CRP_programming_instruction p.5  fused=0.0315  dense=0.759  bm25=22.83]  Page: 5 of 72
Safe operation protocols
★All the operators of robot systems should participate in sys...
[EN-00014-A0_CRP-G4-CD60_Electric_Cabinet_Manual_ p.5  fused=0.0311  dense=0.786  bm25=17.21]  III
Safety Instructions
Brief introduction
This section mainly introduces the safety principles and ...
[EN-00001-A1_Robot_Operation_Manual__1 p.12  fused=0.0299  dense=0.745  bm25=20.49]  dents 
may be caused.
Relevant operators of industrial robots must receive relevant 
professional tr...
[CRP-Welding_Technology_Instruction_Manual p.3  fused=0.0

## 6. Cargar el modelo generador (`google/gemma-7b-it`, base, sin LoRA)

Mismo `BitsAndBytesConfig` (4-bit) que `train_gemma.ipynb` /
`evaluate_gemma.ipynb`, para que el costo de cómputo y la calidad de
generación sean comparables.


In [8]:
!pip install torch


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Modelo base cargado (sin adaptadores LoRA).")


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

d:\OneDrivePersonal\OneDrive\Maestria\Aprendizaje automatico\venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--google--gemma-7b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

## 7. Prompt RAG + función de generación

El contexto recuperado se antepone a la pregunta dentro del mismo mensaje
`user` (la plantilla de chat de Gemma no tiene rol `system`).


In [ ]:
RAG_INSTRUCTION = (
    "You are a technical assistant for CRP robot manuals. "
    "Answer the question using ONLY the context below. "
    "If the answer is not contained in the context, say you don't know. "
    "Be concise and precise.\n\n"
)

def build_rag_prompt(question, k=TOP_K):
    retrieved = retrieve(question, k=k)
    context = "\n\n".join(
        f"[Source: {r['manual']}, page {r['page']}]\n{r['text']}" for r in retrieved
    )
    user_content = f"{RAG_INSTRUCTION}Context:\n{context}\n\nQuestion: {question}"
    return user_content, retrieved

def responder_rag(question, max_new_tokens=MAX_NEW_TOKENS, k=TOP_K):
    user_content, retrieved = build_rag_prompt(question, k=k)
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, retrieved


## 8. Prueba rápida manual (2-3 preguntas)

In [ ]:
for q in [
    "What safety status must the motor be in before operating the robot?",
    "How do you establish the tool coordinate system?",
]:
    answer, retrieved = responder_rag(q)
    print(f"Q: {q}")
    print(f"Fuentes: {[(r['manual'], r['page']) for r in retrieved]}")
    print(f"A: {answer}")
    print("-" * 80)


## 9. Cargar el MISMO split de test que `evaluate_gemma.ipynb`

Se reconstruye con `train_test_split(test_size=EVAL_SPLIT_SIZE, seed=SEED)`
sobre `qa.jsonl` — con la misma semilla, `datasets` devuelve exactamente el
mismo split que se usó para evaluar base vs. afinado.


In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset_split = full_dataset.train_test_split(test_size=EVAL_SPLIT_SIZE, seed=SEED)
test_dataset = dataset_split["test"]

prompts = [ex["messages"][0]["content"] for ex in test_dataset]
references = [ex["messages"][1]["content"] for ex in test_dataset]

print(f"Evaluando RAG sobre {len(test_dataset)} ejemplos (mismo split que evaluate_gemma.ipynb).")


## 10. Generar respuestas RAG para todo el split de test

In [ ]:
preds_rag = []
sources_rag = []

print("Generando respuestas con RAG (modelo base + contexto recuperado)...")
for i, q in enumerate(prompts, 1):
    answer, retrieved = responder_rag(q)
    preds_rag.append(answer)
    sources_rag.append("; ".join(f"{r['manual']} p.{r['page']}" for r in retrieved))
    if i % 10 == 0 or i == len(prompts):
        print(f"  ... RAG: {i}/{len(prompts)}")


## 11. Calcular ROUGE y BERTScore (igual que `evaluate_gemma.ipynb`)

Nota: `evaluate_gemma.ipynb` usa `lang="es"` en BERTScore, pero el dataset
(`qa.jsonl`) está en inglés — probablemente un residuo de copiar la
plantilla del lab de resumen en español. Aquí usamos `lang="en"` (deja que
`evaluate` elija su modelo por defecto para inglés, más preciso que forzar
uno multilingüe). Para que la comparación final sea justa, en la sección 13
recalculamos BERTScore para base/afinado con esta MISMA configuración, en
vez de reusar los números de `evaluate_gemma.ipynb`.


In [ ]:
import evaluate

BERTSCORE_LANG = "en"

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_rag = rouge.compute(predictions=preds_rag, references=references)
rouge_rag_per_example = rouge.compute(predictions=preds_rag, references=references, use_aggregator=False)

bs_rag = bertscore.compute(predictions=preds_rag, references=references, lang=BERTSCORE_LANG)
bertscore_f1_rag = sum(bs_rag["f1"]) / len(bs_rag["f1"])

print("ROUGE (RAG):     ", rouge_rag)
print("BERTScore F1 (RAG):", round(bertscore_f1_rag, 4))


## 12. Guardar el detalle por ejemplo (RAG)

In [ ]:
import pandas as pd

df_rag = pd.DataFrame({
    "pregunta": prompts,
    "referencia": references,
    "respuesta_rag": preds_rag,
    "fuentes_rag": sources_rag,
    "rougeL_rag": rouge_rag_per_example["rougeL"],
})
df_rag.to_csv(OUTPUT_CSV, index=False)
print(f"Detalle RAG guardado en: {OUTPUT_CSV}")


## 13. Tabla comparativa final: base vs. afinado (LoRA) vs. RAG

Se fusiona con `eval_crobotp_base_vs_finetuned.csv` (el CSV que produce
`evaluate_gemma.ipynb`), haciendo match por la columna `pregunta`. Como ese
CSV solo guardó `rougeL` por ejemplo (no ROUGE completo ni BERTScore),
aquí se recalculan ROUGE y BERTScore para `respuesta_base` y
`respuesta_afinada` con la MISMA configuración que usamos para RAG —así
las tres columnas de la tabla final son comparables entre sí, sin depender
de qué `lang`/modelo se haya usado quizás distinto en el otro notebook.


In [ ]:
import os

if not os.path.isfile(FINETUNE_EVAL_CSV):
    print(f"No se encontro {FINETUNE_EVAL_CSV}.")
    print("Corre evaluate_gemma.ipynb primero para poder comparar base/afinado/RAG.")
    print("Mientras tanto, aqui estan los resultados de RAG solo:")
    df_resumen_rag = pd.DataFrame({
        "modelo": ["RAG (base + contexto recuperado)"],
        "rouge1": [rouge_rag["rouge1"]],
        "rouge2": [rouge_rag["rouge2"]],
        "rougeL": [rouge_rag["rougeL"]],
        "rougeLsum": [rouge_rag["rougeLsum"]],
        "bertscore_f1": [bertscore_f1_rag],
        "longitud_promedio_palabras": [sum(len(p.split()) for p in preds_rag) / len(preds_rag)],
    })
else:
    df_ft = pd.read_csv(FINETUNE_EVAL_CSV)

    # Fusionar por pregunta (misma columna en ambos notebooks, mismo split -> deberia calzar 1 a 1)
    df_merged = df_rag.merge(df_ft, on=["pregunta", "referencia"], how="inner")
    print(f"Ejemplos fusionados (deberia ser ~{len(df_rag)}): {len(df_merged)}")
    if len(df_merged) != len(df_rag):
        print("Aviso: no todos los ejemplos calzaron -verifica que ambos notebooks usen "
              "el mismo DATA_PATH, EVAL_SPLIT_SIZE y SEED.")

    # Recalcular ROUGE + BERTScore para base y afinado con la misma config que RAG
    rouge_base_full = rouge.compute(predictions=df_merged["respuesta_base"].tolist(), references=df_merged["referencia"].tolist())
    rouge_ft_full = rouge.compute(predictions=df_merged["respuesta_afinada"].tolist(), references=df_merged["referencia"].tolist())
    rouge_rag_full = rouge.compute(predictions=df_merged["respuesta_rag"].tolist(), references=df_merged["referencia"].tolist())

    bs_base = bertscore.compute(predictions=df_merged["respuesta_base"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)
    bs_ft = bertscore.compute(predictions=df_merged["respuesta_afinada"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)
    bs_rag_full = bertscore.compute(predictions=df_merged["respuesta_rag"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)

    def avg_len(col):
        return sum(len(x.split()) for x in df_merged[col]) / len(df_merged)

    df_resumen_rag = pd.DataFrame({
        "modelo": ["base (sin LoRA, sin RAG)", "afinado (LoRA)", "RAG (base + contexto)"],
        "rouge1": [rouge_base_full["rouge1"], rouge_ft_full["rouge1"], rouge_rag_full["rouge1"]],
        "rouge2": [rouge_base_full["rouge2"], rouge_ft_full["rouge2"], rouge_rag_full["rouge2"]],
        "rougeL": [rouge_base_full["rougeL"], rouge_ft_full["rougeL"], rouge_rag_full["rougeL"]],
        "rougeLsum": [rouge_base_full["rougeLsum"], rouge_ft_full["rougeLsum"], rouge_rag_full["rougeLsum"]],
        "bertscore_f1": [
            sum(bs_base["f1"]) / len(bs_base["f1"]),
            sum(bs_ft["f1"]) / len(bs_ft["f1"]),
            sum(bs_rag_full["f1"]) / len(bs_rag_full["f1"]),
        ],
        "longitud_promedio_palabras": [
            avg_len("respuesta_base"), avg_len("respuesta_afinada"), avg_len("respuesta_rag"),
        ],
    })

    df_merged.to_csv(COMPARISON_CSV, index=False)
    print(f"Detalle fusionado (base/afinado/RAG por ejemplo) guardado en: {COMPARISON_CSV}")

referencia_len = sum(len(r.split()) for r in references) / len(references)
print(f"(Longitud promedio de la referencia: {referencia_len:.1f} palabras)")
df_resumen_rag


## Notas finales

- **Por qué el mismo split:** usar `seed=42` y `test_size=0.10` idénticos
  garantiza que las 977 preguntas de `qa.jsonl` se dividen exactamente
  igual en los tres notebooks, así que las tres filas de la tabla final
  responden a las mismas preguntas sobre las mismas referencias.
- **RAG vs. fine-tuning, qué esperar:** el fine-tuning le enseña al modelo
  el *estilo* de respuesta del dataset (formato, brevedad, tono), mientras
  que el RAG le da *hechos frescos* del corpus sin tocar los pesos del
  modelo. Es normal que RAG gane en preguntas muy específicas de un manual
  (números de parámetro, nombres de instrucciones exactas) y pierda un poco
  en ROUGE si el estilo de redacción del modelo base no calza tan bien con
  las referencias como el modelo afinado.
- **Costo:** el índice FAISS se construye en CPU (gratis); solo la
  generación con Gemma-7b-it usa la GPU L4 que ya tenías reservada para el
  fine-tuning -reutiliza la misma VM/contenedor, no necesitas una segunda.
- **Ampliación opcional:** se puede repetir este mismo notebook cargando el
  modelo **afinado + RAG** (adaptadores LoRA + contexto recuperado) para
  ver si ambas técnicas se complementan -bastaría con cargar el modelo como
  en `evaluate_gemma.ipynb` (con `PeftModel.from_pretrained`) en vez del
  modelo base de la sección 6, y correr el resto del notebook igual.
